In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data")  # notebook lives in notebooks/
COLS = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]

frames = []
for f in sorted(DATA_DIR.glob("*.csv")):
    d = pd.read_csv(f, encoding="latin-1", usecols=lambda c: c in COLS)
    d = d.dropna(subset=["Date", "HomeTeam"])
    d["Date"] = pd.to_datetime(d["Date"], dayfirst=True)
    d["season"] = f.stem
    frames.append(d)

df = pd.concat(frames, ignore_index=True).sort_values("Date").reset_index(drop=True)

C:\Users\oskar\AppData\Local\Temp\ipykernel_42272\843120780.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d["Date"] = pd.to_datetime(d["Date"], dayfirst=True)


In [4]:
# Missing values
print("Missing values:\n", df.isnull().sum())
print("\n" + "=" * 50 + "\n")

# Team name consistency
print("Unique home teams:", df["HomeTeam"].nunique())
print("Unique away teams:", df["AwayTeam"].nunique())
print("\nTeams appearing as home but not away (or vice versa):",
      set(df["HomeTeam"]) ^ set(df["AwayTeam"]))
print("\nHome games per team:\n", df["HomeTeam"].value_counts())
print("\n" + "=" * 50 + "\n")

# Class balance
print("Result distribution:\n", df["FTR"].value_counts())
print("\nResult proportions:\n", df["FTR"].value_counts(normalize=True).round(3))

Missing values:
 Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
B365H       0
B365D       0
B365A       0
season      0
dtype: int64


Unique home teams: 35
Unique away teams: 35

Teams appearing as home but not away (or vice versa): set()

Home games per team:
 HomeTeam
Everton             193
Man City            193
Tottenham           193
Crystal Palace      192
Arsenal             192
Chelsea             192
Man United          192
Liverpool           192
West Ham            190
Brighton            174
Newcastle           174
Bournemouth         155
Burnley             152
Southampton         152
Leicester           152
Wolves              152
Aston Villa         135
Fulham              117
Brentford            98
Watford              95
Leeds                79
Nott'm Forest        79
West Brom            57
Sheffield United     57
Sunderland           40
Swansea              38
Stoke                38
Huddersfield         38
Norwich            

In [5]:
odds_cols = ["B365H", "B365D", "B365A"]
data = df.dropna(subset=odds_cols + ["FTR"]).copy()

y = data["FTR"].map({"H": 0, "D": 1, "A": 2}).to_numpy()

implied = 1 / data[odds_cols].to_numpy()
probs = implied / implied.sum(axis=1, keepdims=True)  # remove bookmaker margin

odds_log_loss = -np.log(probs[np.arange(len(y)), y]).mean()
odds_accuracy = (probs.argmax(axis=1) == y).mean()
home_accuracy = (y == 0).mean()

prior = np.bincount(y) / len(y)
prior_log_loss = -np.log(prior[y]).mean()

print(f"Rows used: {len(y)} (dropped {len(df) - len(y)})")
print(f"Always home:   acc {home_accuracy:.3f}")
print(f"Class priors:  log loss {prior_log_loss:.3f}")
print(f"Bookmaker odds: acc {odds_accuracy:.3f}, log loss {odds_log_loss:.3f}")

Rows used: 3850 (dropped 0)
Always home:   acc 0.445
Class priors:  log loss 1.065
Bookmaker odds: acc 0.555, log loss 0.954
